# Grokfast experiments

CSCI 567 final project: Vishal Patel, Salman Sattar, Tomohiko Uemura, Chinenye Okoro.

This notebook runs every experiment for our study of
[*Grokfast: Accelerated Grokking by Amplifying Slow Gradients*](https://arxiv.org/abs/2405.20233)
on the modular-multiplication task (x·y mod 97). Each experiment is a named grid in
`experiments/sweep.py`; this notebook clones the repo, starts grids in the background,
checks on them, and summarizes the results.

[Open in Colab](https://colab.research.google.com/github/vip11017/csci567-grokfast/blob/salman/notebooks/grokfast_experiments.ipynb)
· use a **GPU runtime (T4 is enough)**: *Runtime → Change runtime type*.

| Section | What it does |
|---|---|
| 1. Setup | Clone the repo, check the GPU, choose where results are saved |
| 2. Helpers | `launch`, `status` and `report` for any grid |
| 3. Replication | Baseline vs. Grokfast-MA vs. Grokfast-EMA, 3 seeds |
| 4. H2: weight decay | Does tuned weight decay alone close the gap? |
| 5. H1: momentum and frequency | Is it ordinary momentum, or the *slow* part of the gradient? |
| 6. Analysis | Tables of steps to 95% validation accuracy |
| 7. Saving results | Keep results after the runtime shuts down |

Background and write-ups live in `docs/` (start with `docs/ideas.md` and `docs/results-replication.md`).

## 1. Setup

Clones the `salman` branch (or pulls it if it's already here) and checks the GPU.

In [ ]:
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/vip11017/csci567-grokfast.git"
BRANCH = "salman"
REPO_DIR = "/content/csci567-grokfast"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

### Where results go

The Colab runtime is wiped when it shuts down. Set `USE_DRIVE = True` to write results
straight to Google Drive (Colab asks for permission once). Otherwise results stay in
`/content/results` until you save them (section 7).

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = "/content/drive/MyDrive/csci567-grokfast"
else:
    RESULTS = "/content/results"
os.makedirs(RESULTS, exist_ok=True)
print("results go to", RESULTS)

## 2. Helpers

`launch(grid)` starts a grid from `experiments/sweep.py` as a background process, so cells
return right away and the runs keep going even if this notebook's kernel restarts.
Relaunching a grid skips runs that already finished.

On a T4, 8 runs in parallel works well (the model is tiny, so most of the cost is Python
overhead). A Grokfast run takes a few minutes; a no-filter baseline can take an hour.

In [ ]:
def launch(grid, parallel=8, seeds=(0, 1, 2), budget=100000):
    """Start a grid in the background. Safe to rerun: finished runs are skipped."""
    out = f"{RESULTS}/{grid}"
    os.makedirs(out, exist_ok=True)
    cmd = [sys.executable, "experiments/sweep.py", "--grid", grid, "--out", out,
           "--parallel", str(parallel), "--budget", str(budget),
           "--seeds", *[str(s) for s in seeds]]
    log = open(f"{out}/sweep.log", "a")
    subprocess.Popen(cmd, cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT,
                     stdin=subprocess.DEVNULL, start_new_session=True)
    print(f"launched {grid} -> {out}")


def status(grid):
    """How many runs are done, running and queued."""
    with open(f"{RESULTS}/{grid}/status.json") as f:
        s = json.load(f)
    print(f"{grid}: {s['done']}/{s['total']} done, {len(s['running'])} running, "
          f"{len(s['queued'])} queued (updated {s['updated']} UTC)")
    for name in s["running"]:
        print("  running:", name)


def report(grid, match=""):
    """One line per run: first step at >= 95% val accuracy, last step, last val accuracy."""
    out = subprocess.run([sys.executable, "experiments/report.py", f"{RESULTS}/{grid}", "--match", match],
                         capture_output=True, text=True)
    print(out.stdout or out.stderr)

## 3. Replication

The paper's headline comparison (Table 3): baseline Adam with no weight decay, Grokfast-MA
(window 100, λ 5, weight decay 0.01) and Grokfast-EMA (α 0.98, λ 2, weight decay 0.005),
for seeds 0–2. Paper: 39,890 / 790 / 910 steps to 95% validation accuracy.

Already run on 2026-10-30. Results are in `results/replication/`, written up in
`docs/results-replication.md` (median 56,800 / 900 / 920 steps).

In [ ]:
# launch("replication")   # ~1.5 h on a T4; the baseline runs dominate
# status("replication")
# report("replication")

## 4. H2: weight decay vs. the filter

The paper's baseline uses no weight decay, while its Grokfast runs use 0.005–0.01. This grid
asks whether tuned weight decay alone explains most of the speedup:

- optimizer: Adam (weight decay added to the gradient, as in the paper's `main.py`) and AdamW (decoupled)
- weight decay: 0, 0.001, 0.005, 0.01, 0.05, 0.1, 1.0
- filter: none, Grokfast-EMA (α 0.98, λ 2)
- seeds 0–2, 75 runs, about 5 hours on a T4

Runs that can't even fit the training set (train accuracy < 50% at step 20,000) are stopped
and counted as failures; this happens with strong decay under Adam.

In [ ]:
launch("h2")

In [ ]:
status("h2")

In [ ]:
report("h2")

## 5. H1: ordinary momentum, or the slow part of the gradient?

Adam already averages gradients (β₁ = 0.9); Grokfast-EMA adds a second, slower average in
front of it. This grid separates the two explanations, with Adam at weight decay 0 and 0.005:

- **filters at the same gain (λ 2, α 0.98):** none, EMA (Grokfast), high-pass EMA
  `g + λ(g − EMA(g))`, and a random direction with the same size as `λ·EMA(g)`
- **more Adam momentum, no filter:** β₁ ∈ {0.95, 0.98, 0.99}
- seeds 0–2, 42 runs

If the slow component is what matters, Grokfast should beat both the high-pass and random
filters, and higher β₁ shouldn't match it.

In [ ]:
launch("h1")

In [ ]:
status("h1")

In [ ]:
report("h1")

## 6. Analysis

Steps to 95% validation accuracy, median over seeds (per-seed values in parentheses).
"fail" means fewer than half the seeds reached the threshold.

In [ ]:
!python experiments/report.py {RESULTS}/h2 > {RESULTS}/h2_report.txt
!python analysis/h2_table.py {RESULTS}/h2_report.txt

In [ ]:
!python analysis/summarize_runs.py {RESULTS}/h1

## 7. Saving results

If `USE_DRIVE` was off, copy everything to Drive before the runtime shuts down:

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !mkdir -p /content/drive/MyDrive/csci567-grokfast && cp -r {RESULTS}/* /content/drive/MyDrive/csci567-grokfast/

Without Drive, `experiments/report.py <folder> --export` prints a compressed copy of all
curves that can be pasted into a file and decoded locally with
`python experiments/report.py --decode <file> --out results/<grid>`.